<a href="https://colab.research.google.com/github/rounak393/clab/blob/main/baseunet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
os.environ["KAGGLE_API_TOKEN"]='KGAT_c03d989b55c966d18c971a92b023645b'

In [2]:
!kaggle datasets download -d britikak/busi-dataset

Dataset URL: https://www.kaggle.com/datasets/britikak/busi-dataset
License(s): unknown
100% 195M/195M [00:02<00:00, 96.8MB/s]



In [3]:

!unzip -q busi-dataset.zip -d busi_dataset

In [7]:
!pip install albumentations scikit-learn -q

import os
import cv2
import torch
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from PIL import Image
import albumentations as A
from sklearn.model_selection import train_test_split
from tqdm import tqdm


BATCH_SIZE = 16
EPOCHS = 100
LR = 1e-3
IMG_SIZE = 256
BASE_DIR = "/content/busi_dataset/Dataset_BUSI_with_GT"
CLASSES = ["benign", "malignant"]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True


train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=15, p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
])

val_test_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
])

class BasicBUSIDataset(Dataset):
    def __init__(self, base_dir, classes, transform=None):
        self.samples = []
        self.transform = transform
        for cls in classes:
            cls_dir = os.path.join(base_dir, cls)
            if not os.path.exists(cls_dir): continue
            images = [f for f in os.listdir(cls_dir) if f.endswith(".png") and "_mask" not in f]
            for img_name in images:
                img_path = os.path.join(cls_dir, img_name)
                base_name = img_name.replace(".png", "")
                mask_files = [f for f in os.listdir(cls_dir) if f.startswith(base_name + "_mask") and f.endswith(".png")]
                if not mask_files: continue
                self.samples.append((img_path, [os.path.join(cls_dir, f) for f in mask_files]))

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        img_path, mask_paths = self.samples[idx]

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)


        combined_mask = np.zeros(image.shape[:2], dtype=np.uint8)
        for mpath in mask_paths:
            mask = np.array(Image.open(mpath).convert("L"))
            combined_mask = np.logical_or(combined_mask, (mask > 0).astype(np.uint8))
        combined_mask = combined_mask.astype(np.float32)

        if self.transform:
            augmented = self.transform(image=image, mask=combined_mask)
            image, combined_mask = augmented["image"], augmented["mask"]

        t_image = torch.from_numpy(image).permute(2, 0, 1).float()
        t_mask = torch.from_numpy(combined_mask).unsqueeze(0).float()

        return t_image, t_mask

full_dataset = BasicBUSIDataset(BASE_DIR, classes=CLASSES, transform=None)
indices = list(range(len(full_dataset)))

train_idx, temp_idx = train_test_split(indices, test_size=0.2, random_state=42)
val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, random_state=42)

train_loader = DataLoader(Subset(BasicBUSIDataset(BASE_DIR, CLASSES, transform=train_transform), train_idx), batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(Subset(BasicBUSIDataset(BASE_DIR, CLASSES, transform=val_test_transform), val_idx), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(Subset(BasicBUSIDataset(BASE_DIR, CLASSES, transform=val_test_transform), test_idx), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


In [8]:

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.conv(x)

class BasicUNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1):
        super().__init__()

        # Encoder
        self.enc1 = DoubleConv(in_channels, 64)
        self.enc2 = DoubleConv(64, 128)
        self.enc3 = DoubleConv(128, 256)
        self.enc4 = DoubleConv(256, 512)
        self.pool = nn.MaxPool2d(2)

        # Bottleneck
        self.bottleneck = DoubleConv(512, 1024)

        # Decoder
        self.up4 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.dec4 = DoubleConv(1024, 512)

        self.up3 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec3 = DoubleConv(512, 256)

        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec2 = DoubleConv(256, 128)

        self.up1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec1 = DoubleConv(128, 64)

        # Final Output Layer
        self.out_conv = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        # Bottleneck
        b = self.bottleneck(self.pool(e4))

        # Decoder Pathway
        d4 = self.dec4(torch.cat([self.up4(b), e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))

        return self.out_conv(d1)

In [9]:
class BCEDiceLoss(nn.Module):
    def __init__(self, smooth=1e-5):
        super().__init__()
        self.smooth = smooth
    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets)
        probs = torch.sigmoid(logits).view(-1)
        targets_f = targets.view(-1)
        inter = (probs * targets_f).sum()
        dice = 1 - (2. * inter + self.smooth) / (probs.sum() + targets_f.sum() + self.smooth)
        return 0.5 * bce + 0.5 * dice

def calc_dice(y_true, logits, smooth=1e-5):
    y_pred = (torch.sigmoid(logits) > 0.5).float().view(-1)
    y_true_f = y_true.view(-1)
    inter = (y_true_f * y_pred).sum()
    return (2. * inter + smooth) / (y_true_f.sum() + y_pred.sum() + smooth)

model = BasicUNet(in_channels=3, out_channels=1).to(device)
criterion = BCEDiceLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scaler = torch.amp.GradScaler('cuda')

best_val_dice = 0.0
best_epoch = 0



for epoch in range(EPOCHS):
    model.train()
    train_loss, train_dice = 0, 0

    for images, masks in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=False):
        images, masks = images.to(device, non_blocking=True), masks.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast('cuda'):
            logits = model(images)
            loss = criterion(logits, masks)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()
        with torch.no_grad():
            train_dice += calc_dice(masks, logits).item()

    model.eval()
    val_loss, val_dice = 0, 0
    with torch.no_grad():
        for images, masks in val_loader:
            images, masks = images.to(device, non_blocking=True), masks.to(device, non_blocking=True)
            with torch.amp.autocast('cuda'):
                logits = model(images)
                loss = criterion(logits, masks)
            val_loss += loss.item()
            val_dice += calc_dice(masks, logits).item()

    avg_t_loss = train_loss / len(train_loader)
    avg_t_dice = train_dice / len(train_loader)
    avg_v_loss = val_loss / len(val_loader)
    avg_v_dice = val_dice / len(val_loader)

    print(f"Epoch [{epoch+1:02d}/{EPOCHS}] | T-Loss: {avg_t_loss:.4f} | T-Dice: {avg_t_dice:.4f} || V-Loss: {avg_v_loss:.4f} | V-Dice: {avg_v_dice:.4f}")

    if avg_v_dice > best_val_dice:
        best_val_dice = avg_v_dice
        best_epoch = epoch + 1
        torch.save(model.state_dict(), "best_basic_unet.pth")


print(f" TRAINING COMPLETE | Best Model saved at Epoch {best_epoch} with V-Dice: {best_val_dice:.4f}")


Epoch [01/100] | T-Loss: 0.6305 | T-Dice: 0.0405 || V-Loss: 0.6006 | V-Dice: 0.0888


Epoch [02/100] | T-Loss: 0.5275 | T-Dice: 0.3465 || V-Loss: 1.0067 | V-Dice: 0.2197


Epoch [03/100] | T-Loss: 0.4563 | T-Dice: 0.4600 || V-Loss: 0.5450 | V-Dice: 0.3316


Epoch [04/100] | T-Loss: 0.4139 | T-Dice: 0.5023 || V-Loss: 0.7053 | V-Dice: 0.2776


Epoch [05/100] | T-Loss: 0.3791 | T-Dice: 0.5403 || V-Loss: 0.5095 | V-Dice: 0.2458


...

Epoch [96/100] | T-Loss: 0.1102 | T-Dice: 0.8739 || V-Loss: 0.2935 | V-Dice: 0.6238


Epoch [97/100] | T-Loss: 0.1123 | T-Dice: 0.8690 || V-Loss: 0.2637 | V-Dice: 0.6834


Epoch [98/100] | T-Loss: 0.1185 | T-Dice: 0.8644 || V-Loss: 0.2802 | V-Dice: 0.6347


Epoch [99/100] | T-Loss: 0.1417 | T-Dice: 0.8390 || V-Loss: 0.3376 | V-Dice: 0.5805


Epoch [100/100] | T-Loss: 0.1179 | T-Dice: 0.8639 || V-Loss: 0.2909 | V-Dice: 0.6540
 TRAINING COMPLETE | Best Model saved at Epoch 81 with V-Dice: 0.7342


In [10]:

model.load_state_dict(torch.load("best_basic_unet.pth", weights_only=True))
model.eval()

test_loss, test_dice = 0, 0

with torch.no_grad():
    for images, masks in tqdm(test_loader, desc="Testing", leave=False):
        images, masks = images.to(device, non_blocking=True), masks.to(device, non_blocking=True)

        with torch.amp.autocast('cuda'):
            logits = model(images)
            loss = criterion(logits, masks)

        test_loss += loss.item()
        test_dice += calc_dice(masks, logits).item()

avg_test_loss = test_loss / len(test_loader)
avg_test_dice = test_dice / len(test_loader)


print(f" FINAL BASIC U-NET TEST LOSS: {avg_test_loss:.4f}")
print(f" FINAL BASIC U-NET TEST DICE: {avg_test_dice:.4f}")


 FINAL BASIC U-NET TEST LOSS: 0.2442
 FINAL BASIC U-NET TEST DICE: 0.7471
